# Yahoo Finance Exploration for Bronze Layer

This notebook helps us understand which Yahoo Finance datasets are useful for the project and how they should land in the bronze layer.

Project goal for this notebook:
- explore daily stock price history
- inspect corporate actions such as dividends and splits
- inspect company metadata useful for dimensions later
- design the raw bronze file layout we will use in S3 later


## Step 1. Choose a small sample universe

For the first exploration, keep the scope small and representative:
- US equities: `AAPL`, `MSFT`, `NVDA`
- Brazil equities: `PETR4.SA`, `VALE3.SA`
- ETF example: `SPY`
- Crypto can be explored later in a separate notebook

This is enough to test different geographies and ticker formats without making the notebook noisy.

In [16]:
from datetime import datetime, timezone
from pathlib import Path
import json

import pandas as pd
import yfinance as yf

In [17]:
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

sample_tickers = ["AAPL", "MSFT", "NVDA", "PETR4.SA", "VALE3.SA", "SPY"]
analysis_end_date = pd.Timestamp.today().normalize()
analysis_start_date = analysis_end_date - pd.Timedelta(days=180)

print("Sample tickers:", sample_tickers)
print("Analysis window:", analysis_start_date.date(), "to", analysis_end_date.date())

Sample tickers: ['AAPL', 'MSFT', 'NVDA', 'PETR4.SA', 'VALE3.SA', 'SPY']
Analysis window: 2025-09-23 to 2026-03-22


## Step 2. Explore daily market prices

The first bronze dataset should be daily OHLCV prices because it supports almost every downstream use case:
- trend analysis
- volatility metrics
- dashboard time series
- joins with company and sector metadata later

We will start by checking what `yfinance` returns for one ticker.

In [18]:
ticker = yf.Ticker("AAPL")

aapl_history = ticker.history(
    start=analysis_start_date.strftime("%Y-%m-%d"),
    end=analysis_end_date.strftime("%Y-%m-%d"),
    interval="1d",
    auto_adjust=False,
    actions=True,
)

aapl_history.head()

DEBUG    Entering history()
DEBUG     Entering history()
DEBUG      AAPL: Yahoo GET parameters: {'period1': '2025-09-23 00:00:00-04:00', 'period2': '2026-03-22 00:00:00-04:00', 'interval': '1d', 'includePrePost': False, 'events': 'div,splits,capitalGains'}
DEBUG      Entering get()
DEBUG       Entering _make_request()
DEBUG        url=https://query2.finance.yahoo.com/v8/finance/chart/AAPL
DEBUG        params=frozendict.frozendict({'period1': 1758600000, 'period2': 1774152000, 'interval': '1d', 'includePrePost': False, 'events': 'div,splits,capitalGains'})
DEBUG        Entering _get_cookie_and_crumb()
DEBUG         cookie_mode = 'csrf'
DEBUG         Entering _get_crumb_csrf()
DEBUG          reusing crumb
DEBUG         Exiting _get_crumb_csrf()
DEBUG        Exiting _get_cookie_and_crumb()
DEBUG        response code=200
DEBUG       Exiting _make_request()
DEBUG      Exiting get()
DEBUG      AAPL: yfinance received OHLC data: 2025-09-23 13:30:00 -> 2026-03-20 20:00:00
DEBUG      AAPL: OHLC

,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2025-09-23 00:00:00-04:00,255.880005,257.339996,253.580002,254.429993,253.945969,60275200,0.0,0.0
2025-09-24 00:00:00-04:00,255.220001,255.740005,251.039993,252.309998,251.830002,42303700,0.0,0.0
2025-09-25 00:00:00-04:00,253.210007,257.170013,251.710007,256.869995,256.381317,55202100,0.0,0.0
2025-09-26 00:00:00-04:00,254.100006,257.600006,253.779999,255.460007,254.974014,46076300,0.0,0.0
2025-09-29 00:00:00-04:00,254.559998,255.000000,253.009995,254.429993,253.945969,40127700,0.0,0.0


In [20]:
print("Shape:", aapl_history.shape)
print("Index type:", type(aapl_history.index))
print("Columns:", list(aapl_history.columns))
print("Null counts:")
aapl_history.isna().sum()

Shape: (124, 8)
Index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
Columns: ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Dividends', 'Stock Splits']
Null counts:


Open            0
High            0
Low             0
Close           0
Adj Close       0
Volume          0
Dividends       0
Stock Splits    0
dtype: int64

## Step 3. Download multiple tickers

For the pipeline, we will ingest multiple tickers per run. Here we test the multi-ticker output and then reshape it into a normalized format.

In [21]:
raw_multi_prices = yf.download(
    tickers=sample_tickers,
    start=analysis_start_date.strftime("%Y-%m-%d"),
    end=analysis_end_date.strftime("%Y-%m-%d"),
    interval="1d",
    auto_adjust=False,
    actions=True,
    group_by="ticker",
    progress=False,
)

raw_multi_prices.head()

DEBUG    Entering download()
DEBUG     Disabling multithreading because DEBUG logging enabled
DEBUG     Entering history()
DEBUG      Entering history()
DEBUG       MSFT: Yahoo GET parameters: {'period1': '2025-09-23 00:00:00-04:00', 'period2': '2026-03-22 00:00:00-04:00', 'interval': '1d', 'includePrePost': False, 'events': 'div,splits,capitalGains'}
DEBUG       Entering get()
DEBUG        Entering _make_request()
DEBUG         url=https://query2.finance.yahoo.com/v8/finance/chart/MSFT
DEBUG         params=frozendict.frozendict({'period1': 1758600000, 'period2': 1774152000, 'interval': '1d', 'includePrePost': False, 'events': 'div,splits,capitalGains'})
DEBUG         Entering _get_cookie_and_crumb()
DEBUG          cookie_mode = 'csrf'
DEBUG          Entering _get_crumb_csrf()
DEBUG           reusing crumb
DEBUG          Exiting _get_crumb_csrf()
DEBUG         Exiting _get_cookie_and_crumb()
DEBUG         response code=200
DEBUG        Exiting _make_request()
DEBUG       Exiting get()


Ticker            MSFT                                                                                            SPY              \
Price             Open        High         Low       Close   Adj Close      Volume Dividends Stock Splits        Open        High   
Date                                                                                                                                
2025-09-23  513.799988  514.590027  507.309998  509.230011  507.121246  19799600.0       0.0          0.0  666.719971  667.340027   
2025-09-24  510.380005  512.479980  506.920013  510.149994  508.037384  13533700.0       0.0          0.0  664.510010  664.609985   
2025-09-25  508.299988  510.010010  505.040009  507.029999  504.930328  15786500.0       0.0          0.0  657.940002  659.409973   
2025-09-26  510.059998  513.940002  506.619995  511.459991  509.341949  16213100.0       0.0          0.0  659.510010  662.369995   
2025-09-29  511.500000  516.849976  508.880005  514.599976  512.468933  17617800.0       0.0          0.0  664.359985  665.280029   

Ticker                                                                                            VALE3.SA                        \
Price              Low       Close   Adj Close      Volume Dividends Stock Splits Capital Gains       Open       High        Low   
Date                                                                                                                               
2025-09-23  661.979980  663.210022  659.455139  81708900.0       0.0          0.0           0.0  58.000000  58.389999  57.660000   
2025-09-24  659.669983  661.099976  657.356995  68082200.0       0.0          0.0           0.0  57.950001  58.139999  57.669998   
2025-09-25  654.409973  658.049988  654.324280  89622100.0       0.0          0.0           0.0  58.189999  58.549999  58.040001   
2025-09-26  657.880005  661.820007  658.072998  69179200.0       0.0          0.0           0.0  57.740002  57.939999  56.540001   
2025-09-29  661.859985  663.679993  659.922424  73499000.0       0.0          0.0           0.0  57.500000  57.750000  57.130001   

Ticker                                                                     NVDA                                                  \
Price           Close  Adj Close      Volume Dividends Stock Splits        Open        High         Low       Close   Adj Close   
Date                                                                                                                              
2025-09-23  57.669998  54.801098  17652400.0       0.0          0.0  181.970001  182.419998  176.210007  178.429993  178.410400   
2025-09-24  57.889999  55.010155  13005700.0       0.0          0.0  179.770004  179.779999  175.399994  176.970001  176.950562   
2025-09-25  58.209999  55.314236  13631100.0       0.0          0.0  174.479996  180.259995  173.130005  177.690002  177.670502   
2025-09-26  57.090000  54.249954  29037000.0       0.0          0.0  178.169998  179.770004  174.929993  178.190002  178.170441   
2025-09-29  57.279999  54.430500  15580200.0       0.0          0.0  180.429993  184.000000  180.320007  181.850006  181.830048   

Ticker                                                AAPL                                                                        \
Price            Volume Dividends Stock Splits        Open        High         Low       Close   Adj Close      Volume Dividends   
Date                                                                                                                               
2025-09-23  192559600.0       0.0          0.0  255.880005  257.339996  253.580002  254.429993  253.945969  60275200.0       0.0   
2025-09-24  143564100.0       0.0          0.0  255.220001  255.740005  251.039993  252.309998  251.830002  42303700.0       0.0   
2025-09-25  191586700.0       0.0          0.0  253.210007  257.170013  251.710007  256.869995  256.381317  55202100.0       0.0   
2025-09-26  148573700.0       0.0    

In [22]:
normalized_price_frames = []

for symbol in sample_tickers:
    if symbol not in raw_multi_prices.columns.get_level_values(0):
        continue

    symbol_df = raw_multi_prices[symbol].copy()
    symbol_df = symbol_df.reset_index().rename(columns={"Date": "price_date"})
    symbol_df["symbol"] = symbol
    normalized_price_frames.append(symbol_df)

prices_daily = pd.concat(normalized_price_frames, ignore_index=True)
prices_daily.columns = [str(col).strip().lower().replace(" ", "_") for col in prices_daily.columns]

ordered_columns = [
    "symbol",
    "price_date",
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume",
    "dividends",
    "stock_splits",
]

existing_columns = [column for column in ordered_columns if column in prices_daily.columns]
prices_daily = prices_daily[existing_columns]
prices_daily.head(10)

,symbol,price_date,open,high,low,close,adj_close,volume,dividends,stock_splits
0,AAPL,2025-09-23,255.880005,257.339996,253.580002,254.429993,253.945969,60275200.0,0.0,0.0
1,AAPL,2025-09-24,255.220001,255.740005,251.039993,252.309998,251.830002,42303700.0,0.0,0.0
2,AAPL,2025-09-25,253.210007,257.170013,251.710007,256.869995,256.381317,55202100.0,0.0,0.0
3,AAPL,2025-09-26,254.100006,257.600006,253.779999,255.460007,254.974014,46076300.0,0.0,0.0
4,AAPL,2025-09-29,254.559998,255.000000,253.009995,254.429993,253.945969,40127700.0,0.0,0.0
5,AAPL,2025-09-30,254.860001,255.919998,253.110001,254.630005,254.145599,37704300.0,0.0,0.0
6,AAPL,2025-10-01,255.039993,258.790009,254.929993,255.449997,254.964035,48713900.0,0.0,0.0
7,AAPL,2025-10-02,256.579987,258.179993,254.149994,257.130005,256.640839,42630200.0,0.0,0.0
8,AAPL,2025-10-03,254.669998,259.239990,253.949997,258.019989,257.529144,49155600.0,0.0,0.0
9,AAPL,2025-10-06,257.989990,259.070007,255.050003,256.690002,256.201691,44664100.0,0.0,0.0


In [23]:
print("Row count:", len(prices_daily))
print("Distinct symbols:", prices_daily["symbol"].nunique())
print("Date range:", prices_daily["price_date"].min(), "to", prices_daily["price_date"].max())
print("Duplicate business key count:", prices_daily.duplicated(subset=["symbol", "price_date"]).sum())

prices_daily.isna().sum()

Row count: 756
Distinct symbols: 6
Date range: 2025-09-23 00:00:00 to 2026-03-20 00:00:00
Duplicate business key count: 0


symbol           0
price_date       0
open            16
high            16
low             16
close           16
adj_close       16
volume          16
dividends       16
stock_splits    16
dtype: int64

## Step 4. Explore corporate actions

Dividends and stock splits are important because they explain differences between raw close and adjusted close. They are also useful datasets on their own.

In [24]:
aapl_actions = ticker.actions.copy()
aapl_dividends = ticker.dividends.copy()
aapl_splits = ticker.splits.copy()

print("Actions shape:", aapl_actions.shape)
display(aapl_actions.tail(10))

print("Dividends shape:", aapl_dividends.shape)
display(aapl_dividends.tail(10))

print("Splits shape:", aapl_splits.shape)
display(aapl_splits.tail(10))

DEBUG    Entering history()
DEBUG     AAPL: Yahoo GET parameters: {'period1': '1927-04-16 21:51:53-05:00', 'period2': '2026-03-22 22:51:48-04:00', 'interval': '1d', 'includePrePost': True, 'events': 'div,splits,capitalGains'}
DEBUG     Entering get()
DEBUG      Entering _make_request()
DEBUG       url=https://query2.finance.yahoo.com/v8/finance/chart/AAPL
DEBUG       params={'period1': -1347829687, 'period2': 1774234308, 'interval': '1d', 'includePrePost': True, 'events': 'div,splits,capitalGains'}
DEBUG       Entering _get_cookie_and_crumb()
DEBUG        cookie_mode = 'csrf'
DEBUG        Entering _get_crumb_csrf()
DEBUG         reusing crumb
DEBUG        Exiting _get_crumb_csrf()
DEBUG       Exiting _get_cookie_and_crumb()
DEBUG       response code=200
DEBUG      Exiting _make_request()
DEBUG     Exiting get()
DEBUG     AAPL: yfinance received OHLC data: 1980-12-12 14:30:00 -> 2026-03-20 20:00:00
DEBUG     AAPL: OHLC after cleaning: 1980-12-12 09:30:00-05:00 -> 2026-03-20 16:00:00-04:

Actions shape: (95, 2)


,Dividends,Stock Splits
Date,,
2023-11-10 00:00:00-05:00,0.24,0.0
2024-02-09 00:00:00-05:00,0.24,0.0
2024-05-10 00:00:00-04:00,0.25,0.0
2024-08-12 00:00:00-04:00,0.25,0.0
2024-11-08 00:00:00-05:00,0.25,0.0
2025-02-10 00:00:00-05:00,0.25,0.0
2025-05-12 00:00:00-04:00,0.26,0.0
2025-08-11 00:00:00-04:00,0.26,0.0
2025-11-10 00:00:00-05:00,0.26,0.0


Dividends shape: (90,)


Date
2023-11-10 00:00:00-05:00    0.24
2024-02-09 00:00:00-05:00    0.24
2024-05-10 00:00:00-04:00    0.25
2024-08-12 00:00:00-04:00    0.25
2024-11-08 00:00:00-05:00    0.25
2025-02-10 00:00:00-05:00    0.25
2025-05-12 00:00:00-04:00    0.26
2025-08-11 00:00:00-04:00    0.26
2025-11-10 00:00:00-05:00    0.26
2026-02-09 00:00:00-05:00    0.26
Name: Dividends, dtype: float64

Splits shape: (5,)


Date
1987-06-16 00:00:00-04:00    2.0
2000-06-21 00:00:00-04:00    2.0
2005-02-28 00:00:00-05:00    2.0
2014-06-09 00:00:00-04:00    7.0
2020-08-31 00:00:00-04:00    4.0
Name: Stock Splits, dtype: float64

## Step 5. Explore company metadata

This is useful for later building dimensions such as `dim_company`, `dim_sector`, and dashboard filters.

Important note:
- metadata fields can vary by ticker
- some fields may be missing
- we should land the raw payload in bronze first and only normalize stable fields in silver

In [25]:
ticker.fast_info

lazy-loading dict with keys = ['currency', 'dayHigh', 'dayLow', 'exchange', 'fiftyDayAverage', 'lastPrice', 'lastVolume', 'marketCap', 'open', 'previousClose', 'quoteType', 'regularMarketPreviousClose', 'shares', 'tenDayAverageVolume', 'threeMonthAverageVolume', 'timezone', 'twoHundredDayAverage', 'yearChange', 'yearHigh', 'yearLow']

In [26]:
aapl_info = ticker.info

selected_info_fields = {
    key: aapl_info.get(key)
    for key in [
        "symbol",
        "shortName",
        "longName",
        "quoteType",
        "sector",
        "industry",
        "country",
        "currency",
        "exchange",
        "marketCap",
        "enterpriseValue",
        "fullTimeEmployees",
    ]
}

pd.Series(selected_info_fields)

DEBUG    get_raw_json(): https://query2.finance.yahoo.com/v10/finance/quoteSummary/AAPL
DEBUG    Entering get()
DEBUG     Entering _make_request()
DEBUG      url=https://query2.finance.yahoo.com/v10/finance/quoteSummary/AAPL
DEBUG      params={'modules': 'financialData,quoteType,defaultKeyStatistics,assetProfile,summaryDetail', 'corsDomain': 'finance.yahoo.com', 'formatted': 'false', 'symbol': 'AAPL'}
DEBUG      Entering _get_cookie_and_crumb()
DEBUG       cookie_mode = 'csrf'
DEBUG       Entering _get_crumb_csrf()
DEBUG        reusing crumb
DEBUG       Exiting _get_crumb_csrf()
DEBUG      Exiting _get_cookie_and_crumb()
DEBUG      response code=401
DEBUG      toggling cookie strategy csrf -> basic
DEBUG      Entering _get_cookie_and_crumb()
DEBUG       cookie_mode = 'basic'
DEBUG       Entering _get_cookie_and_crumb_basic()
DEBUG        Entering _get_cookie_basic()
DEBUG         Entering _load_cookie_curlCffi()
DEBUG         Exiting _load_cookie_curlCffi()
DEBUG         reusing persis

symbol                               AAPL
shortName                      Apple Inc.
longName                       Apple Inc.
quoteType                          EQUITY
sector                         Technology
industry             Consumer Electronics
country                     United States
currency                              USD
exchange                              NMS
marketCap                   3644938780672
enterpriseValue             3664377806848
fullTimeEmployees                  150000
dtype: object

In [28]:
metadata_rows = []

for symbol in sample_tickers:
    symbol_ticker = yf.Ticker(symbol)
    info = symbol_ticker.info
    metadata_rows.append(
        {
            "symbol": symbol,
            "short_name": info.get("shortName"),
            "long_name": info.get("longName"),
            "quote_type": info.get("quoteType"),
            "sector": info.get("sector"),
            "industry": info.get("industry"),
            "country": info.get("country"),
            "currency": info.get("currency"),
            "exchange": info.get("exchange"),
            "market_cap": info.get("marketCap"),
        }
    )

company_metadata = pd.DataFrame(metadata_rows)
company_metadata

DEBUG    get_raw_json(): https://query2.finance.yahoo.com/v10/finance/quoteSummary/AAPL
DEBUG    Entering get()
DEBUG     Entering _make_request()
DEBUG      url=https://query2.finance.yahoo.com/v10/finance/quoteSummary/AAPL
DEBUG      params={'modules': 'financialData,quoteType,defaultKeyStatistics,assetProfile,summaryDetail', 'corsDomain': 'finance.yahoo.com', 'formatted': 'false', 'symbol': 'AAPL'}
DEBUG      Entering _get_cookie_and_crumb()
DEBUG       cookie_mode = 'basic'
DEBUG       Entering _get_cookie_and_crumb_basic()
DEBUG        Entering _get_cookie_basic()
DEBUG         reusing cookie
DEBUG        Exiting _get_cookie_basic()
DEBUG        Entering _get_crumb_basic()
DEBUG         reusing crumb
DEBUG        Exiting _get_crumb_basic()
DEBUG       Exiting _get_cookie_and_crumb_basic()
DEBUG      Exiting _get_cookie_and_crumb()
DEBUG      response code=200
DEBUG     Exiting _make_request()
DEBUG    Exiting get()
DEBUG    get_raw_json(): https://query1.finance.yahoo.com/v7/finan

,symbol,short_name,long_name,quote_type,sector,industry,country,currency,exchange,market_cap
0,AAPL,Apple Inc.,Apple Inc.,EQUITY,Technology,Consumer Electronics,United States,USD,NMS,3644938780672
1,MSFT,Microsoft Corporation,Microsoft Corporation,EQUITY,Technology,Software - Infrastructure,United States,USD,NMS,2838053519360
2,NVDA,NVIDIA Corporation,NVIDIA Corporation,EQUITY,Technology,Semiconductors,United States,USD,NMS,4203063541760
3,PETR4.SA,PETROBRAS PN N2,Petróleo Brasileiro S.A. - Petrobras,EQUITY,Energy,Oil & Gas Integrated,Brazil,BRL,SAO,588628426752
4,VALE3.SA,VALE ON NM,Vale S.A.,EQUITY,Basic Materials,Other Industrial Metals & Mining,Brazil,BRL,SAO,322506358784
5,SPY,State Street SPDR S&P 500 ETF T,State Street SPDR S&P 500 ETF Trust,ETF,None,None,None,USD,PCX,595245858816


## Step 6. Bronze layer design

For the bronze layer, keep data as raw as possible.

Recommended raw datasets from Yahoo Finance:
- `market_prices_daily`
- `corporate_actions`
- `company_metadata`

Recommended partition pattern in S3:

`s3://financial-data/bronze/source=yahoo_finance/dataset=market_prices_daily/symbol=AAPL/ingestion_date=2026-03-22/file.json`

Important metadata to store with each bronze record:
- source
- dataset
- symbol
- ingestion timestamp
- request parameters
- raw payload

In [29]:
ingestion_ts = datetime.now(timezone.utc).isoformat()

bronze_market_prices_payload = {
    "source": "yahoo_finance",
    "dataset": "market_prices_daily",
    "symbol": "AAPL",
    "ingestion_ts_utc": ingestion_ts,
    "request_params": {
        "start": analysis_start_date.strftime("%Y-%m-%d"),
        "end": analysis_end_date.strftime("%Y-%m-%d"),
        "interval": "1d",
        "auto_adjust": False,
        "actions": True,
    },
    "raw_payload": aapl_history.reset_index().to_dict(orient="records"),
}

print(json.dumps(bronze_market_prices_payload, indent=2, default=str)[:2500])

{
  "source": "yahoo_finance",
  "dataset": "market_prices_daily",
  "symbol": "AAPL",
  "ingestion_ts_utc": "2026-03-23T02:53:37.928942+00:00",
  "request_params": {
    "start": "2025-09-23",
    "end": "2026-03-22",
    "interval": "1d",
    "auto_adjust": false,
    "actions": true
  },
  "raw_payload": [
    {
      "Date": "2025-09-23 00:00:00-04:00",
      "Open": 255.8800048828125,
      "High": 257.3399963378906,
      "Low": 253.5800018310547,
      "Close": 254.42999267578125,
      "Adj Close": 253.9459686279297,
      "Volume": 60275200,
      "Dividends": 0.0,
      "Stock Splits": 0.0
    },
    {
      "Date": "2025-09-24 00:00:00-04:00",
      "Open": 255.22000122070312,
      "High": 255.74000549316406,
      "Low": 251.0399932861328,
      "Close": 252.30999755859375,
      "Adj Close": 251.8300018310547,
      "Volume": 42303700,
      "Dividends": 0.0,
      "Stock Splits": 0.0
    },
    {
      "Date": "2025-09-25 00:00:00-04:00",
      "Open": 253.2100067138672,

In [30]:
ingestion_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

bronze_relative_path = Path(
    f"bronze/source=yahoo_finance/dataset=market_prices_daily/symbol=AAPL/ingestion_date={ingestion_date}/aapl_prices.json"
)

bronze_relative_path.as_posix()

'bronze/source=yahoo_finance/dataset=market_prices_daily/symbol=AAPL/ingestion_date=2026-03-23/aapl_prices.json'

## Step 7. What we learned from this notebook

Best Yahoo Finance datasets for this project:
- daily OHLCV prices
- dividends and stock splits
- company metadata for dimensions

Best use of Yahoo Finance in your architecture:
- bronze: land raw market and metadata payloads
- silver: normalize prices and metadata into tabular datasets
- gold: combine with Alpha Vantage fundamentals and macro indicators for analytics

Suggested next notebook:
- `02_alpha_vantage_exploration.ipynb`

There we should explore:
- daily adjusted stock prices
- company overview
- income statement
- balance sheet
- FX daily
- one macro indicator such as GDP